[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/06_information_theory_in_deep_learning/exercises.ipynb)

# Exercises: Information Theory in Deep Learning

**20 fully solved problems** in 4 levels: Concept Check (4), Foundation (6), Applications in AI/ML (6), Challenge (4).

## Level 0 — Concept Check

### Problem L0.1: The ELBO Gap

A VAE reports $\mathcal{L}_{\mathrm{ELBO}} = -104.2$ nats on an example whose true log-likelihood under the model is $\log p_\theta(x) = -101.7$ nats. What is the inference gap, and what does it measure?

**Solution**

By the exact decomposition,

$$
\log p_\theta(x) = \mathcal{L}_{\mathrm{ELBO}} + D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right)
$$

so the gap is

$$
D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right) = -101.7 - (-104.2) = 2.5 \text{ nats}
$$

This is *not* model error — it is **inference** error: the amount by which the amortized encoder's approximate posterior misses the model's true posterior. Improving the decoder cannot reduce it; a richer variational family (flows, importance-weighted bounds, more encoder capacity) can.

$$
\boxed{\text{gap} = D_{\mathrm{KL}}\left(q_\phi \parallel p_\theta(\cdot \mid x)\right) = 2.5 \text{ nats}}
$$

*Key takeaway*: The ELBO's slack is always a KL divergence, which tells you exactly which component to fix — encoder for the gap, decoder and prior for the bound itself.

### Problem L0.2: Reading the Information Bottleneck Multiplier

In $\mathcal{L}_{\mathrm{IB}} = I(X; Z) - \beta I(Z; Y)$ (minimized), describe the optimal representation as $\beta \to 0$ and as $\beta \to \infty$.

**Solution**

**$\beta \to 0$.** The objective reduces to minimizing $I(X; Z)$ alone. The minimum is $I(X; Z) = 0$, achieved by any $Z$ independent of $X$ — a constant. The representation is maximally compressed and completely useless.

**$\beta \to \infty$.** Relevance dominates, so the optimum maximizes $I(Z; Y)$ irrespective of cost. By the data-processing inequality $I(Z; Y) \le I(X; Y)$, so the optimum achieves $I(Z; Y) = I(X; Y)$: $Z$ becomes a **minimal sufficient statistic** of $X$ for $Y$ (minimal because among all sufficient $Z$ the rate term still breaks ties at any large finite $\beta$).

**In between.** $\beta$ traces the information curve, and its reciprocal is the curve's slope: $\frac{dI(Z; Y)}{dI(X; Z)} = \frac{1}{\beta}$ at the optimum.

$$
\boxed{\beta \to 0: \; Z \perp X \text{ (zero rate)}; \qquad \beta \to \infty: \; Z \text{ minimal sufficient for } Y}
$$

*Key takeaway*: $\beta$ is an exchange rate in "nats of relevance per nat of rate"; the two limits bracket every useful representation.

### Problem L0.3: Chance-Level InfoNCE

A contrastive model with batch size $K = 128$ reports $\mathcal{L}_{\mathrm{NCE}} = 4.85$ nats. Is it learning anything?

**Solution**

Chance level is uniform over the $K$ candidates, giving loss $\log K$:

$$
\log 128 = 4.852 \text{ nats}
$$

The reported $4.85$ is indistinguishable from chance. The certified bound is

$$
I(X; Y) \ge \log K - \mathcal{L}_{\mathrm{NCE}} = 4.852 - 4.85 \approx 0.002 \text{ nats}
$$

i.e., essentially nothing. Common causes: the critic collapsed (all embeddings identical), the temperature is mis-set, positives and negatives are mislabeled in the batch construction, or the two views share no information.

$$
\boxed{\mathcal{L}_{\mathrm{NCE}} \approx \log K \implies \text{certified } I \approx 0}
$$

*Key takeaway*: Always compare a contrastive loss to $\log K$, never to zero — the same discipline as comparing a classification loss to $\log(\text{number of classes})$.

### Problem L0.4: Two-Part Codes and Model Selection

Model A needs 200 bits to describe and codes the dataset in 8,400 bits. Model B needs 1,500 bits to describe and codes it in 7,000 bits. Which does MDL prefer, and what is the equivalent likelihood-ratio statement?

**Solution**

Total description lengths:

$$
L_A = 200 + 8{,}400 = 8{,}600 \text{ bits}, \qquad L_B = 1{,}500 + 7{,}000 = 8{,}500 \text{ bits}
$$

MDL prefers **model B** by 100 bits: its extra 1,300 bits of complexity buy 1,400 bits of data compression.

**Likelihood-ratio reading.** $L(\mathcal{D} \mid M) = -\log_2 p(\mathcal{D} \mid M)$, so B's data term corresponds to a likelihood ratio of $2^{1400}$ in B's favor, while the complexity terms act as a prior ratio $2^{-1300}$. The comparison is exactly a Bayes factor with prior $\pi(M) \propto 2^{-L(M)}$:

$$
\log_2\frac{p(\mathcal{D} \mid B)\pi(B)}{p(\mathcal{D} \mid A)\pi(A)} = 1400 - 1300 = 100 \text{ bits}
$$

$$
\boxed{L_B = 8{,}500 \lt L_A = 8{,}600 \text{ bits} \implies \text{choose B}}
$$

*Key takeaway*: MDL is Bayesian model selection with the prior written as a code; "complexity penalty" and "prior" are the same object in different units.

## Level 1 — Foundation

### Problem L1.1: Derive the ELBO Two Ways

Derive $\log p_\theta(x) \ge \mathbb{E}_{q_\phi}\left[\log p_\theta(x \mid z)\right] - D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)$ (a) by Jensen's inequality and (b) by an exact identity, and state when the bound is tight.

**Solution**

**(a) Jensen.** Introduce $q_\phi$ inside the marginalization:

$$
\log p_\theta(x) = \log \mathbb{E}_{q_\phi(z \mid x)}\left[\frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right] \ge \mathbb{E}_{q_\phi}\left[\log\frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right]
$$

Factoring $p_\theta(x, z) = p_\theta(x \mid z)p(z)$ splits the right side into the reconstruction term and $-D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)$.

**(b) Exact identity.** Factor the other way, $p_\theta(x, z) = p_\theta(z \mid x)p_\theta(x)$:

$$
\mathbb{E}_{q_\phi}\left[\log\frac{p_\theta(x, z)}{q_\phi(z \mid x)}\right] = \log p_\theta(x) - \mathbb{E}_{q_\phi}\left[\log\frac{q_\phi(z \mid x)}{p_\theta(z \mid x)}\right] = \log p_\theta(x) - D_{\mathrm{KL}}\left(q_\phi \parallel p_\theta(\cdot \mid x)\right)
$$

No inequality was used; the bound follows from $D_{\mathrm{KL}} \ge 0$.

**Tightness.** The bound is an equality iff $q_\phi(z \mid x) = p_\theta(z \mid x)$ almost everywhere. Route (b) is the more useful derivation because it *names* the slack.

$$
\boxed{\log p_\theta(x) - \mathcal{L}_{\mathrm{ELBO}} = D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p_\theta(z \mid x)\right) \ge 0}
$$

*Key takeaway*: Jensen tells you a bound exists; the exact identity tells you what it costs, and every variational method in this module follows the second pattern.

### Problem L1.2: Gaussian KL and the Arithmetic of Posterior Collapse

For a diagonal Gaussian posterior $\mathcal{N}(\mu, \mathrm{diag}(\sigma^2))$ against prior $\mathcal{N}(0, I)$, write the per-dimension KL, find where it vanishes, and compute it for $(\mu_j, \sigma_j) = (0, 0.99)$ and $(2, 1)$.

**Solution**

**Formula.**

$$
D_j = \tfrac{1}{2}\left(\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2\right)
$$

**Zero set.** $\partial D_j/\partial \mu_j = \mu_j = 0$ and $\partial D_j/\partial\sigma_j^2 = \tfrac{1}{2}\left(1 - 1/\sigma_j^2\right) = 0$ give the unique stationary point $(\mu_j, \sigma_j^2) = (0, 1)$, where $D_j = 0$. Since $D_j$ is a KL it is nonnegative, so this is the global minimum — a dimension carrying literally no information.

**Numerics.**

- $(0, 0.99)$: $D_j = \tfrac{1}{2}\left(0 + 0.9801 - 1 - \ln 0.9801\right) = \tfrac{1}{2}\left(-0.0199 + 0.0201\right) \approx 0.0001$ nats — collapsed for all practical purposes.
- $(2, 1)$: $D_j = \tfrac{1}{2}\left(4 + 1 - 1 - 0\right) = 2$ nats — a well-used dimension.

**Second-order picture.** Near the minimum, writing $\sigma_j^2 = 1 + \delta$, $D_j \approx \tfrac{1}{2}\mu_j^2 + \tfrac{1}{4}\delta^2$: the penalty is quadratic in both deviations, so gradients vanish at collapse and the optimizer has no pressure to escape.

$$
\boxed{D_j = \tfrac{1}{2}\left(\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2\right), \quad D_j = 0 \iff (\mu_j, \sigma_j) = (0, 1)}
$$

*Key takeaway*: Posterior collapse is a *flat, attracting* optimum of the rate term; "free bits" and KL annealing exist to keep the optimizer away from it early in training.

### Problem L1.3: The Rate Decomposition

Prove $\mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)\right] = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right)$ and interpret each term.

**Solution**

**Step 1 (insert the aggregate posterior $q(z) = \mathbb{E}_{p(x)}\left[q(z \mid x)\right]$).**

$$
\log\frac{q(z \mid x)}{p(z)} = \log\frac{q(z \mid x)}{q(z)} + \log\frac{q(z)}{p(z)}
$$

**Step 2 (average the first term under $p(x)q(z \mid x)$).** This is by definition the mutual information of the joint $p(x)q(z \mid x)$:

$$
\mathbb{E}\left[\log\frac{q(z \mid x)}{q(z)}\right] = I(X; Z)
$$

**Step 3 (average the second term).** It depends only on $z$, whose marginal is exactly $q(z)$, so it averages to $D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right)$.

**Step 4 (interpret).**

- $I(X; Z)$ is the information the code actually carries about the data — the quantity we care about.
- $D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right)$ is *pure waste*: bits spent because the code's actual marginal does not match the prior used to encode it (the "prior hole"). Learned priors, VampPrior, and normalizing-flow priors exist to shrink it.

$$
\boxed{R = I(X; Z) + D_{\mathrm{KL}}\left(q(z) \parallel p(z)\right) \ge I(X; Z)}
$$

*Key takeaway*: The VAE's KL term is an upper bound on the latent's mutual information, and the excess is a fixable modeling inefficiency rather than an information cost.

### Problem L1.4: $\beta$-VAE as Constrained Optimization

Show that maximizing the ELBO subject to a rate constraint $R \le R_0$ is equivalent, by Lagrangian duality, to maximizing $\mathbb{E}_q\left[\log p(x \mid z)\right] - \beta R$, and interpret $\beta$.

**Solution**

**Step 1 (the constrained problem).**

$$
\max_{\phi, \theta} \; -D \quad \text{subject to} \quad R \le R_0
$$

where $D = \mathbb{E}\left[-\log p_\theta(x \mid z)\right]$ and $R = \mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)\right]$.

**Step 2 (Lagrangian).**

$$
\mathcal{G}(\phi, \theta, \beta) = -D - \beta\left(R - R_0\right), \qquad \beta \ge 0
$$

Since $\beta R_0$ is constant in the parameters, maximizing $\mathcal{G}$ is maximizing $-D - \beta R$ — the $\beta$-VAE objective, with $\beta = 1$ recovering the plain ELBO.

**Step 3 (interpretation of $\beta$).** At the optimum, complementary slackness ties $\beta$ to the constraint: $\beta \gt 0$ implies $R = R_0$, and the envelope theorem gives

$$
\beta = -\frac{\partial D^{*}}{\partial R_0}
$$

i.e., $\beta$ is the *marginal distortion saved per extra nat of rate* — the slope of the rate–distortion curve.

**Step 4 (practical consequence).** Because the curve is convex and decreasing, large $\beta$ selects low-rate points (compressed, blurry, often disentangled) and small $\beta$ selects high-rate points (sharp, entangled). Constrained formulations that target $R_0$ directly with an adaptive $\beta$ (as in GECO or KL-budget controllers) are usually easier to tune than a fixed $\beta$.

$$
\boxed{\max\left\{-D - \beta R\right\} \equiv \max\left\{-D : R \le R_0\right\}, \quad \beta = -\frac{\partial D^{*}}{\partial R_0}}
$$

*Key takeaway*: $\beta$ is not a mysterious knob — it is the Lagrange multiplier of a bit budget, and its optimal value is the local slope of the rate–distortion frontier.

### Problem L1.5: Bits-Back — Why the KL Term Is Real Bits

Show that a bits-back scheme transmits $x$ at expected cost $\mathbb{E}_q\left[-\log p(x \mid z)\right] + D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)$ nats, and explain where the "refund" comes from.

**Solution**

**Step 1 (naive cost).** Sender picks $z \sim q(z \mid x)$, encodes it under the shared prior at cost $-\log p(z)$, then encodes $x$ under $p(x \mid z)$ at cost $-\log p(x \mid z)$:

$$
C_{\text{naive}} = \mathbb{E}_q\left[-\log p(z) - \log p(x \mid z)\right]
$$

**Step 2 (the refund).** The sender does not need fresh coin flips to draw $z$: it can *decode* $z$ from auxiliary bits already waiting in the message queue, using $q(z \mid x)$ as the decoder. Consuming those bits costs nothing extra because they are real payload. After the receiver reconstructs $x$, it can reconstruct $q(z \mid x)$, re-encode $z$ under it, and recover

$$
\text{refund} = \mathbb{E}_q\left[-\log q(z \mid x)\right] \text{ nats}
$$

of genuine message.

**Step 3 (net).**

$$
C = C_{\text{naive}} - \text{refund} = \mathbb{E}_q\left[-\log p(x \mid z)\right] + \mathbb{E}_q\left[\log\frac{q(z \mid x)}{p(z)}\right] = D + D_{\mathrm{KL}}\left(q(z \mid x) \parallel p(z)\right)
$$

which is exactly $-\mathcal{L}_{\mathrm{ELBO}}$.

**Step 4 (optimality).** Since $-\mathcal{L}_{\mathrm{ELBO}} \ge -\log p(x)$, the scheme is within the inference gap of the Shannon optimum $-\log p(x)$, and it is realizable in practice with asymmetric numeral systems (Bits-Back with ANS).

$$
\boxed{\text{bits-back cost} = D + R = -\mathcal{L}_{\mathrm{ELBO}} \ge -\log p(x)}
$$

*Key takeaway*: The KL term is not a metaphor for complexity — a working codec charges exactly those nats, and the ELBO is the code length it achieves.

### Problem L1.6: The Optimal Contrastive Critic

Derive the minimizer of the InfoNCE loss over all functions $f(x, y)$ and explain what a trained critic's logits estimate.

**Solution**

**Step 1 (the task is a $K$-way classification).** Given $x$ and candidates $y_{1:K}$ with exactly one positive, the true posterior over the positive index is

$$
\Pr\left[j \mid x, y_{1:K}\right] \propto p(y_j \mid x)\prod_{k \neq j}p(y_k) \propto \frac{p(y_j \mid x)}{p(y_j)}
$$

after dividing by the common factor $\prod_k p(y_k)$.

**Step 2 (properness of log loss).** InfoNCE is the cross-entropy between this true posterior and the model's softmax $\frac{e^{f(x, y_j)}}{\sum_k e^{f(x, y_k)}}$. By strict propriety of the logarithmic score (Topic 03), the loss is minimized exactly when the two agree:

$$
\frac{e^{f(x, y_j)}}{\sum_k e^{f(x, y_k)}} = \frac{r(x, y_j)}{\sum_k r(x, y_k)}, \qquad r(x, y) = \frac{p(y \mid x)}{p(y)}
$$

**Step 3 (solve).** Softmax is invariant to additive shifts in its argument, so the solutions are

$$
f^{*}(x, y) = \log\frac{p(y \mid x)}{p(y)} + c(x) = \mathrm{PMI}(x, y) + c(x)
$$

**Step 4 (what logits mean).** A trained CLIP-style scaled cosine similarity therefore approximates pointwise mutual information up to a per-anchor offset and the temperature scale: differences of logits for two candidates estimate $\log\frac{p(y_1 \mid x)p(y_2)}{p(y_2 \mid x)p(y_1)}$, a calibrated relevance ratio, while absolute logit values are not comparable across anchors.

$$
\boxed{f^{*}(x, y) = \mathrm{PMI}(x, y) + c(x)}
$$

*Key takeaway*: Contrastive training is density-ratio estimation; that is why its scores retrieve well and why they need per-anchor normalization before being read as probabilities.

## Level 2 — Applications in AI/ML

### Problem L2.1: The RLHF KL Budget

An RLHF run uses $\beta = 0.1$ and converges with a measured $D_{\mathrm{KL}}\left(\pi_\theta \parallel \pi_{\text{ref}}\right) = 8$ nats per response and a mean reward gain of $1.2$ over the reference. (a) Write the optimal policy. (b) Interpret 8 nats. (c) What happens if $\beta$ is halved?

**Solution**

**(a) Optimal policy.** Maximizing $\mathbb{E}_{\pi}\left[r\right] - \beta D_{\mathrm{KL}}\left(\pi \parallel \pi_{\text{ref}}\right)$ gives the exponential tilt

$$
\pi^{*}(y \mid x) = \frac{\pi_{\text{ref}}(y \mid x)\exp\left(r(x, y)/\beta\right)}{Z(x)}, \qquad Z(x) = \mathbb{E}_{\pi_{\text{ref}}}\left[e^{r/\beta}\right]
$$

with optimal objective value $\beta\log Z(x)$ — a free energy.

**(b) Reading 8 nats.** $8$ nats $= 11.5$ bits: a response from the tuned model would need about 11.5 extra bits to encode under the reference model than under itself. Equivalently, the typical likelihood ratio is $e^{8} \approx 3000$; by Pinsker the two policies are separated by at most $\mathrm{TV} \le \sqrt{8/2} = 2$, which is vacuous — at this KL the distributions are effectively disjoint in their typical sets, so "the tuned model writes noticeably different text" is the correct interpretation.

**(c) Halving $\beta$.** The tilt exponent $r/\beta$ doubles, sharpening the optimum toward high-reward responses. The realized KL rises (typically much more than $2\times$, since the objective's marginal trade rate is $\beta$), and the risk of reward hacking — exploiting reward-model errors in low-$\pi_{\text{ref}}$ regions — rises with it. The standard remedy is an adaptive controller that adjusts $\beta$ to hold a target KL budget rather than fixing $\beta$.

$$
\boxed{\pi^{*} \propto \pi_{\text{ref}}e^{r/\beta}; \quad 8 \text{ nats} \approx 11.5 \text{ bits of divergence from the reference}}
$$

*Key takeaway*: The KL penalty defines the optimum rather than merely restraining it; monitoring KL in nats gives a unit-ful, model-independent measure of how far alignment has moved the policy.

### Problem L2.2: The Deep Variational Information Bottleneck

Show how the intractable IB objective $I(X; Z) - \beta I(Z; Y)$ becomes a trainable loss, identifying the bound used on each term and the direction of each bound.

**Solution**

**Step 1 (upper-bound the rate).** By the rate decomposition (Problem L1.3), for any variational prior $p(z)$,

$$
I(X; Z) \le \mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)\right] = R
$$

An *upper* bound is exactly what is needed, because $I(X; Z)$ is being **minimized**.

**Step 2 (lower-bound the relevance).** By Barber–Agakov, for any variational decoder $q_\psi(y \mid z)$,

$$
I(Z; Y) = H(Y) - H(Y \mid Z) \ge H(Y) + \mathbb{E}_{p(x,y)q_\phi(z \mid x)}\left[\log q_\psi(y \mid z)\right]
$$

with slack $\mathbb{E}\left[D_{\mathrm{KL}}\left(p(y \mid z) \parallel q_\psi(y \mid z)\right)\right] \ge 0$. A *lower* bound is what is needed, because $I(Z; Y)$ is being **maximized**. $H(Y)$ is a constant of the dataset and drops out.

**Step 3 (assemble the loss).** Minimize over $(\phi, \psi)$:

$$
\mathcal{L}_{\mathrm{VIB}} = \underbrace{\mathbb{E}\left[-\log q_\psi(y \mid z)\right]}_{\text{ordinary cross-entropy}} + \frac{1}{\beta}\,\mathbb{E}_{p(x)}\left[D_{\mathrm{KL}}\left(q_\phi(z \mid x) \parallel p(z)\right)\right]
$$

This is a classifier with a stochastic bottleneck layer and a Gaussian KL penalty — a one-line change to a standard training script.

**Step 4 (why the bound directions matter).** Both bounds are *conservative in the right direction*, so $\mathcal{L}_{\mathrm{VIB}}$ upper-bounds the true IB objective; minimizing a valid upper bound is a sound relaxation. Reversing either bound would make the surrogate meaningless.

**Step 5 (empirical effects).** Capping the rate improves calibration and adversarial robustness (an attacker must move $x$ far enough to move a low-capacity $Z$) at a small cost in clean accuracy.

$$
\boxed{\mathcal{L}_{\mathrm{VIB}} = \mathbb{E}\left[-\log q_\psi(y \mid z)\right] + \tfrac{1}{\beta}\,R \; \ge \; \text{IB objective (up to a constant)}}
$$

*Key takeaway*: Variational information objectives are assembled by bounding each mutual information in the direction its optimization requires — the single most transferable trick in this module.

### Problem L2.3: Contrastive Batch Size, Temperature, and the Ceiling

A SimCLR-style run uses $K = 1024$ in-batch negatives and temperature $\tau = 0.1$. (a) What is the MI ceiling? (b) How does $\tau$ enter the bound? (c) What does raising $K$ to $65{,}536$ via a memory bank buy?

**Solution**

**(a) Ceiling.** $\log 1024 = 6.93$ nats $= 10$ bits. No matter how good the encoder, InfoNCE at this batch size can certify at most 10 bits of mutual information between the two views.

**(b) Temperature.** The critic is $f(x, y) = \mathrm{sim}(x, y)/\tau$. Because the optimal critic is $\mathrm{PMI} + c(x)$ (Problem L1.6), $\tau$ sets the scale on which the cosine similarity is interpreted as a log density ratio: the model can only represent PMI values within roughly $\left[-2/\tau, 2/\tau\right]$ given bounded cosine similarities. Small $\tau$ widens that range and sharpens the softmax, up-weighting hard negatives; too small and gradients concentrate on a few pairs and training destabilizes. $\tau$ does *not* change the $\log K$ ceiling.

**(c) Larger $K$.** Raising $K$ to $65{,}536$ lifts the ceiling to $\log 65{,}536 = 11.09$ nats $= 16$ bits. The gain is logarithmic: a $64\times$ larger negative pool buys $\log 64 = 4.16$ nats $= 6$ bits. Memory banks and momentum encoders (MoCo) obtain this without a $64\times$ larger gradient batch, which is the entire engineering point.

$$
\boxed{\text{ceiling} = \log K: \; 6.93 \text{ nats at } K{=}1024, \; 11.09 \text{ nats at } K{=}65536}
$$

*Key takeaway*: Batch size sets what the objective *can* certify, temperature sets how the critic *represents* density ratios; conflating the two leads to tuning the wrong knob.

### Problem L2.4: Scaling Laws Read as Compression

A model family fits $L(N) = \frac{A}{N^{\alpha}} + L_{\infty}$ with $L$ in nats per token. Measurements give $L = 2.20$ nats at $N = 10^9$ and $L = 1.95$ nats at $N = 10^{10}$; suppose the fitted $L_{\infty} = 1.60$ nats. Compute $\alpha$, the bits-per-token improvement, and the compression consequence.

**Solution**

**Step 1 (subtract the floor).** The reducible parts are $2.20 - 1.60 = 0.60$ and $1.95 - 1.60 = 0.35$ nats.

**Step 2 (solve for $\alpha$).** From $\frac{A}{N^{\alpha}}$ evaluated at the two sizes,

$$
\frac{0.35}{0.60} = \left(\frac{10^{10}}{10^{9}}\right)^{-\alpha} = 10^{-\alpha} \implies \alpha = -\log_{10}(0.5833) = 0.234
$$

**Step 3 (bits per token).** $2.20/\ln 2 = 3.174$ bits and $1.95/\ln 2 = 2.813$ bits: a saving of $0.361$ bits per token.

**Step 4 (compression).** Paired with an arithmetic coder, the larger model compresses a 1-trillion-token corpus by $0.361 \times 10^{12}$ bits $\approx 45$ TB less output — the same corpus, $11\%$ smaller.

**Step 5 (the floor).** $L_{\infty} = 1.60$ nats $= 2.31$ bits per token is an estimate of the entropy rate of the data: even an infinitely large model cannot go below it. Using the Topic 03 identity $H(p, q) = H(p) + D_{\mathrm{KL}}(p \parallel q)$, scaling shrinks only the KL term; at $N = 10^{10}$ the model still pays $0.35$ nats of pure model error.

$$
\boxed{\alpha \approx 0.234; \; \text{saving } 0.36 \text{ bits/token}; \; \text{irreducible floor } L_{\infty} = 2.31 \text{ bits/token}}
$$

*Key takeaway*: Scaling laws are compression curves whose asymptote is the data's entropy; quoting a loss without its floor hides how much of the remaining error is even addressable.

### Problem L2.5: InfoGAN's Mutual-Information Regularizer

InfoGAN adds $\lambda I\left(c; G(z, c)\right)$ to the GAN objective, where $c$ is a latent code. Derive the variational lower bound actually optimized, and explain why the auxiliary network is needed.

**Solution**

**Step 1 (the obstacle).** $I(c; X)$ with $X = G(z, c)$ requires the posterior $p(c \mid x)$, which is unavailable: the generator induces it implicitly and it has no closed form.

**Step 2 (Barber–Agakov).** Introduce an auxiliary network $Q_\psi(c \mid x)$ and write

$$
I(c; X) = H(c) - H(c \mid X) = H(c) + \mathbb{E}_{x \sim G}\mathbb{E}_{c \sim p(c \mid x)}\left[\log p(c \mid x)\right]
$$

Adding and subtracting $\log Q_\psi(c \mid x)$ gives

$$
I(c; X) = H(c) + \mathbb{E}_{x}\left[D_{\mathrm{KL}}\left(p(c \mid x) \parallel Q_\psi(c \mid x)\right)\right] + \mathbb{E}_{c, x}\left[\log Q_\psi(c \mid x)\right] \ge H(c) + \mathbb{E}_{c, x}\left[\log Q_\psi(c \mid x)\right]
$$

**Step 3 (make it samplable).** The remaining expectation still appears to need $p(c \mid x)$, but the standard lemma $\mathbb{E}_{x \sim G}\mathbb{E}_{c' \sim p(c \mid x)}\left[g(c', x)\right] = \mathbb{E}_{c \sim p(c)}\mathbb{E}_{x \sim G(z, c)}\left[g(c, x)\right]$ lets us sample $c$ from the *prior* and $x$ from the generator. The trainable bound is

$$
L_I(G, Q) = H(c) + \mathbb{E}_{c \sim p(c),\, x \sim G(z, c)}\left[\log Q_\psi(c \mid x)\right] \le I(c; X)
$$

With $p(c)$ fixed, $H(c)$ is a constant, so the term reduces to a reconstruction loss on the code: cross-entropy for categorical $c$, Gaussian NLL for continuous $c$.

**Step 4 (why $Q$ is needed).** $Q_\psi$ *is* the variational posterior; its accuracy determines the bound's tightness, and its gradient is what forces the generator to make $c$ recoverable from the image — the mechanism that produces axis-aligned, interpretable factors.

$$
\boxed{L_I = H(c) + \mathbb{E}\left[\log Q_\psi(c \mid x)\right] \le I(c; X), \; \text{gap} = \mathbb{E}\left[D_{\mathrm{KL}}\left(p(c \mid x) \parallel Q_\psi\right)\right]}
$$

*Key takeaway*: "Maximize mutual information with the latent code" is implemented as "make the code decodable from the output" — the Barber–Agakov bound is what licenses that translation.

### Problem L2.6: Weight Noise as Description Length

Hinton and van Camp minimize $\mathbb{E}_{q(w)}\left[-\log p(\mathcal{D} \mid w)\right] + D_{\mathrm{KL}}\left(q(w) \parallel p(w)\right)$ over a Gaussian posterior on weights. Show this is a description length, and compute the KL for one weight with $q = \mathcal{N}(\mu, \sigma^2)$, $p = \mathcal{N}(0, s^2)$.

**Solution**

**Step 1 (it is bits-back applied to weights).** Replace the latent $z$ of Problem L1.5 by the weight vector $w$. A sender transmitting $\mathcal{D}$ can encode $w$ under the shared prior $p(w)$, encode the data under $p(\mathcal{D} \mid w)$, and take back $-\log q(w)$ nats. The net cost is exactly

$$
C = \mathbb{E}_{q}\left[-\log p(\mathcal{D} \mid w)\right] + D_{\mathrm{KL}}\left(q(w) \parallel p(w)\right)
$$

the variational free energy. So "noisy weights with a KL penalty" is literally "describe the model in as few bits as possible, then describe the data given it".

**Step 2 (the per-weight KL).**

$$
D_{\mathrm{KL}}\left(\mathcal{N}(\mu, \sigma^2) \parallel \mathcal{N}(0, s^2)\right) = \ln\frac{s}{\sigma} + \frac{\sigma^2 + \mu^2}{2s^2} - \frac{1}{2}
$$

A weight that is left at the prior ($\mu = 0$, $\sigma = s$) costs 0 nats; a precisely-tuned weight ($\sigma \ll s$) costs about $\ln(s/\sigma)$ nats — precision is paid for in bits.

**Step 3 (consequences).** Pruning, quantization, and weight decay all reappear as ways of reducing this cost: a pruned weight costs nothing, a coarsely quantized weight costs few bits, and $L_2$ regularization is the $\mu^2/2s^2$ term with $\sigma$ held fixed.

**Step 4 (generalization).** PAC-Bayes bounds have the form

$$
\text{risk} \le \widehat{\text{risk}} + \sqrt{\frac{D_{\mathrm{KL}}\left(q(w) \parallel p(w)\right) + \ln(n/\delta)}{2n}}
$$

so the same KL that measures description length also controls the generalization gap — the formal version of "simpler models generalize".

$$
\boxed{D_{\mathrm{KL}} = \ln\frac{s}{\sigma} + \frac{\sigma^2 + \mu^2}{2s^2} - \frac{1}{2} \text{ nats per weight}}
$$

*Key takeaway*: The number of *bits of precision* a network actually needs — not its parameter count — is what MDL and PAC-Bayes charge for, which is why heavily overparameterized but compressible networks generalize.

## Level 3 — Challenge

### Problem L3.1: Deriving the IB Self-Consistent Equations

Derive the stationarity condition of $\mathcal{L} = I(X; Z) - \beta I(Z; Y)$ with respect to $p(z \mid x)$, subject to normalization, and interpret the resulting encoder.

**Solution**

**Step 1 (Lagrangian).** Work in nats and include multipliers for $\sum_z p(z \mid x) = 1$:

$$
\mathcal{F} = I(X; Z) - \beta I(Z; Y) + \sum_x \lambda(x)\sum_z p(z \mid x)
$$

**Step 2 (rate term).** With $I(X; Z) = \sum_{x,z}p(x)p(z \mid x)\ln\frac{p(z \mid x)}{p(z)}$ and $p(z) = \sum_{x'}p(x')p(z \mid x')$,

$$
\frac{\partial I(X; Z)}{\partial p(z \mid x)} = p(x)\left[\ln\frac{p(z \mid x)}{p(z)} + 1\right] - p(x)
$$

the last term arising from the $p(z)$ dependence and cancelling the $+1$.

**Step 3 (relevance term).** Using $I(Z; Y) = \sum_{y,z}p(y, z)\ln\frac{p(y \mid z)}{p(y)}$ with $p(y, z) = \sum_x p(x)p(y \mid x)p(z \mid x)$ (Markov chain $Y \to X \to Z$), and noting that the derivative of $p(y \mid z)$ terms contributes a total that vanishes by normalization,

$$
\frac{\partial I(Z; Y)}{\partial p(z \mid x)} = p(x)\sum_y p(y \mid x)\ln\frac{p(y \mid z)}{p(y)}
$$

**Step 4 (set to zero and identify the KL).** Stationarity gives

$$
\ln\frac{p(z \mid x)}{p(z)} = \beta\sum_y p(y \mid x)\ln\frac{p(y \mid z)}{p(y)} - \tilde\lambda(x)
$$

Write the sum as

$$
\sum_y p(y \mid x)\ln\frac{p(y \mid z)}{p(y)} = -D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right) + \sum_y p(y \mid x)\ln\frac{p(y \mid x)}{p(y)}
$$

The second piece depends only on $x$ and is absorbed into $\tilde\lambda(x)$.

**Step 5 (exponentiate).**

$$
p(z \mid x) = \frac{p(z)}{Z(x, \beta)}\exp\left(-\beta\, D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right)\right)
$$

coupled with $p(z) = \sum_x p(x)p(z \mid x)$ and $p(y \mid z) = \frac{1}{p(z)}\sum_x p(x)p(y \mid x)p(z \mid x)$. Alternating these three updates is the IB algorithm, a Blahut–Arimoto variant that converges to a stationary point.

$$
\boxed{p(z \mid x) \propto p(z)\exp\left(-\beta D_{\mathrm{KL}}\left(p(y \mid x) \parallel p(y \mid z)\right)\right)}
$$

*Key takeaway*: The optimal bottleneck is a soft clustering at inverse temperature $\beta$, assigning $x$ to the cluster whose predictive distribution best matches $x$'s own — and as $\beta$ increases, clusters split at genuine phase transitions.

### Problem L3.2: The Information Curve Is Concave and Its Slope Is $1/\beta$

Let $\mathcal{I}(R) = \max\left\{I(Z; Y) : I(X; Z) \le R\right\}$. Prove that $\mathcal{I}$ is nondecreasing and concave, that it saturates at $I(X; Y)$, and that the optimal $\beta$ satisfies $\mathcal{I}'(R) = 1/\beta$.

**Solution**

**Step 1 (nondecreasing).** The feasible set for $R_2 \gt R_1$ contains that for $R_1$, so the maximum cannot decrease.

**Step 2 (saturation).** The Markov chain $Y \to X \to Z$ and the data-processing inequality give $I(Z; Y) \le I(X; Y)$ for every encoder, so $\mathcal{I}(R) \le I(X; Y)$; taking $Z = X$ (rate $R = H(X)$) attains it.

**Step 3 (concavity).** Let $p_1(z \mid x)$ and $p_2(z \mid x)$ achieve $(R_1, \mathcal{I}(R_1))$ and $(R_2, \mathcal{I}(R_2))$ with disjoint codebooks $\mathcal{Z}_1, \mathcal{Z}_2$. Build a *time-sharing* encoder that uses $p_1$ with probability $\lambda$ and $p_2$ with probability $1 - \lambda$, appending the choice to the code. Mutual information is linear in the mixing weight for such a construction (the extra indicator variable $S$ contributes $I(X; S) = 0$ since $S$ is independent of $X$, and $I(Z, S; Y) = \lambda I_1 + (1-\lambda)I_2$ by conditioning on $S$), so the pair

$$
\left(\lambda R_1 + (1-\lambda)R_2, \; \lambda\mathcal{I}(R_1) + (1-\lambda)\mathcal{I}(R_2)\right)
$$

is achievable. Hence $\mathcal{I}\left(\lambda R_1 + (1-\lambda)R_2\right) \ge \lambda\mathcal{I}(R_1) + (1-\lambda)\mathcal{I}(R_2)$: concavity.

**Step 4 (the slope).** Concavity means the constrained problem $\max\{I(Z; Y) : I(X; Z) \le R\}$ has no duality gap and is equivalent to the Lagrangian form $\max\left\{I(Z; Y) - \frac{1}{\beta}I(X; Z)\right\}$, i.e., to minimizing $\mathcal{L}_{\mathrm{IB}} = I(X; Z) - \beta I(Z; Y)$ up to the factor $\beta$. The envelope theorem gives

$$
\mathcal{I}'(R) = \frac{1}{\beta}
$$

so sweeping $\beta$ from $\infty$ down to 1 traces the curve from the high-rate saturated end toward the origin. Because the curve lies below the diagonal $\mathcal{I}(R) \le R$ (DPI plus $I(Z;Y) \le I(X;Z)$ is *not* generally true, but $\mathcal{I}(R) \le \min(R', I(X;Y))$ holds along the relevant branch), values $\beta \le 1$ yield only the trivial solution $I(X; Z) = 0$.

$$
\boxed{\mathcal{I} \text{ concave, nondecreasing, } \mathcal{I}(\infty) = I(X; Y), \; \mathcal{I}'(R) = 1/\beta}
$$

*Key takeaway*: The IB curve is a rate–distortion curve in disguise; its concavity is what makes $\beta$ a well-behaved dial and what guarantees that every point of the frontier is reachable by some Lagrangian weight.

### Problem L3.3: Why Every $K$-Sample MI Lower Bound Is Capped

Prove that $\mathcal{L}_{\mathrm{NCE}} \ge 0$ in expectation, hence $\log K$ caps the InfoNCE bound, and sketch why the limitation is fundamental rather than an artifact of this particular bound.

**Solution**

**Step 1 (the loss is a cross-entropy).** $\mathcal{L}_{\mathrm{NCE}}$ is the expected negative log-probability that the model assigns to the true index among $K$ candidates:

$$
\mathcal{L}_{\mathrm{NCE}} = \mathbb{E}\left[-\log \hat{P}\left[\text{true index}\right]\right] \ge 0
$$

since $\hat{P} \in (0, 1]$. Therefore $\log K - \mathcal{L}_{\mathrm{NCE}} \le \log K$ for every critic and every dataset.

**Step 2 (the bound is tight only up to the ceiling).** With the optimal critic the bound becomes $\min\left(I(X; Y), \approx \log K\right)$: when the true MI is far below $\log K$ the bound is informative, and when it is far above, the loss saturates near 0 and the bound stalls at $\log K$.

**Step 3 (why this is fundamental).** Consider two joint distributions on $K$-sample datasets: (i) $P_{\text{ind}}$ under which $x$ and $y$ are independent, so $I = 0$; and (ii) $P_{\text{dep}}$, a "hidden matching" construction in which pairs are genuinely dependent but the dependence is carried by an event of probability $e^{-I}$ per sample. The total-variation distance between the two $K$-sample distributions is $O\left(K e^{-I}\right)$.

Any estimator $\hat{I}(x_{1:K}, y_{1:K})$ that is a valid *high-confidence lower bound* must report near 0 under $P_{\text{ind}}$. If it reported more than $\log K + c$ under $P_{\text{dep}}$, it would distinguish two distributions whose total variation is $O(Ke^{-I}) \ll 1$ once $I \gg \log K$ — impossible.

**Step 4 (consequence).** Sample-based MI *lower* bounds are capped at $O(\log K)$ nats, while *upper* bounds (such as the rate $R$ of Problem L1.3) have no such limitation because they use a known prior rather than samples. This asymmetry is why practitioners can honestly report a rate but not an MI.

$$
\boxed{\mathcal{L}_{\mathrm{NCE}} \ge 0 \implies \text{bound} \le \log K; \text{ any } K\text{-sample lower bound is } O(\log K)}
$$

*Key takeaway*: The ceiling is a property of the *estimation problem*, not of InfoNCE; it is why "we maximize mutual information" is a statement about an objective and never about a measurement.

### Problem L3.4: The ELBO Isoline and the Anatomy of Posterior Collapse

Using $D \ge H - R$, explain why a VAE with a powerful autoregressive decoder can attain the optimal ELBO with $R = 0$, and derive the condition under which increasing decoder capacity causes collapse.

**Solution**

**Step 1 (the feasible region).** From Problem L1.3 and the Barber–Agakov bound,

$$
H - D \le I(X; Z) \le R \quad \Longrightarrow \quad D + R \ge H
$$

where $H$ is the data entropy. The ELBO is $-\left(D + R\right)$, so **every** point on the line $D + R = H$ achieves the same, optimal, ELBO value $-H$.

**Step 2 (two extreme optima).**

- *Autoencoding corner*: $R = H$, $D = 0$. The latent carries the whole message; the decoder is nearly deterministic given $z$.
- *Collapse corner*: $R = 0$, $D = H$. The latent is independent of $x$ ($I(X; Z) \le R = 0$), and the decoder alone models $p(x)$ perfectly, paying exactly the entropy.

Both are ELBO-optimal. The objective is *indifferent* between a representation and no representation at all.

**Step 3 (the collapse condition).** Let $D_{\min}(R)$ be the best distortion achievable at rate $R$ by the given decoder family. Collapse occurs when the decoder can reach $D_{\min}(0) = H$ on its own, i.e., when

$$
\mathbb{E}\left[-\log p_\theta(x \mid z)\right] \text{ at } z \perp x \;\approx\; H
$$

An autoregressive decoder $p_\theta(x \mid z) = \prod_t p_\theta(x_t \mid x_{\lt t}, z)$ can model $p(x)$ arbitrarily well while ignoring $z$, so as its capacity grows $D_{\min}(0) \to H$ and the zero-rate corner becomes globally optimal. Since the KL term's gradient at $(\mu, \sigma) = (0, 1)$ is zero (Problem L1.2), the optimizer has no restoring force once it arrives.

**Step 4 (remedies, each breaking a step of the argument).**

- **$\beta \lt 1$ or free bits**: change the objective so the isoline is no longer flat, making a positive rate strictly optimal.
- **Weakened decoder** (limited context, dropout on $x_{\lt t}$): raise $D_{\min}(0)$ above $H$ so the collapse corner is infeasible.
- **KL annealing**: start at $\beta \approx 0$ so the encoder becomes informative before the rate is priced.
- **Explicit rate target** (constrained optimization): fix $R = R_0 \gt 0$ and let the multiplier adapt.

$$
\boxed{D + R \ge H, \text{ ELBO constant on } D + R = H; \text{ collapse} \iff D_{\min}(0) = H}
$$

*Key takeaway*: Posterior collapse is not an optimization bug but a property of the objective's level sets — the ELBO scores compression of the data, and is silent about whether the compression lives in the code or in the decoder.